# Fix Minnie spatial (discardable)

Run top to bottom to repair Minnie65 materialization 1822 spatial coordinates in the configured CCM output root. Requires an existing CAVE token, `standard_transform`, the existing release, and the cached neuron roster from the original ETL.

This copies the nucleus query/deduplication and CCM spatial write from the main notebooks. The transform already returns microns: do not divide by 1,000. The corrected full nucleus-position cache is also saved so future ETL runs retain the fix.

The write keeps both reference-space IDs and uses merge/upsert, not append. Before any writes, the incoming spatial keys must exactly match the existing rows in these two spaces. Other projects/spaces are untouched. Delta retains older versions for time travel, but current reads see only the updated rows. This notebook can be deleted after a successful repair.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import standard_transform
from caveclient import CAVEclient

from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import read_spatial_locations, write_models
from connects_common_connectivity.models import (
    AnatomicalOrientation, Default2DView, LocationType, ReferenceSpace,
    SignedAxis, SpatialLocation, Unit,
)

REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
DATA_ROOT = REPO_ROOT / "data/EM minnie 1822"
settings = get_settings()
OUTPUT_ROOT = Path(settings.output_root)
if settings.dry_run:
    raise RuntimeError("This repair requires dry_run: false; no changes were made.")
PROJECT_ID = "minnie65"
CAVE_REFERENCE_SPACE_ID = "minnie65_v1822_cave_nm"
STANDARD_REFERENCE_SPACE_ID = "minnie65_v1822_standard_transform_um"
print(f"Repair target: {OUTPUT_ROOT}")

Repair target: /workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v6


In [3]:
client = CAVEclient("minnie65_phase3_v1", version=1822)
nuc_df = client.materialize.query_table(
    "nucleus_detection_v0",
    split_positions=True,
    desired_resolution=[1, 1, 1],
)[["id", "pt_position_x", "pt_position_y", "pt_position_z", "pt_root_id", "volume"]]
nuc_df = nuc_df[nuc_df["pt_root_id"] != 0]

pairs = nuc_df[["id", "pt_root_id"]].drop_duplicates()
roots_per_nucleus = pairs.groupby("id")["pt_root_id"].nunique()
nuclei_per_root = pairs.groupby("pt_root_id")["id"].nunique()
ambiguous = nuc_df["id"].isin(
    roots_per_nucleus.index[roots_per_nucleus > 1]
) | nuc_df["pt_root_id"].isin(nuclei_per_root.index[nuclei_per_root > 1])
print(f"dropped {int(ambiguous.sum())} / {len(nuc_df)}")
nuc_df = nuc_df[~ambiguous].reset_index(drop=True)

tform = standard_transform.minnie_transform_nm()
xyz = tform.apply(nuc_df[["pt_position_x", "pt_position_y", "pt_position_z"]].values)
nuc_df[["x_medial-lateral", "y_dorsal-ventral", "z_caudal-rostral"]] = xyz
assert np.isfinite(xyz).all(), "Transform returned non-finite coordinates."
assert nuc_df["id"].is_unique and nuc_df["pt_root_id"].is_unique

neuron_nuclei = pd.read_parquet(DATA_ROOT / "minnie_v1822_neuron_nuclei.parquet")
assert neuron_nuclei["cell_type"].eq("neuron").all()
positions = nuc_df.loc[nuc_df["id"].isin(set(neuron_nuclei["id"]))].copy()
positions["dataitem_id"] = positions["pt_root_id"].astype(str)
assert not positions.empty, "No single-soma neuron positions to repair."
print(positions[["x_medial-lateral", "y_dorsal-ventral", "z_caudal-rostral"]].agg(["min", "max"]))

dropped 11351 / 129815
     x_medial-lateral  y_dorsal-ventral  z_caudal-rostral
min        166.617657         -3.372603            594.40
max       1617.981345        891.945557           1113.84


In [4]:
default_view = Default2DView(
    left_to_right=SignedAxis.PLUS_X,
    bottom_to_top=SignedAxis.MINUS_Y,
)
reference_spaces = [
    ReferenceSpace(
        id=CAVE_REFERENCE_SPACE_ID,
        name="Minnie65 materialization 1822 CAVE coordinates",
        description="CAVE nucleus positions requested at 1 nm resolution.",
        unit=Unit.NANOMETERS_LENGTH,
        default_2d_view=default_view,
        project_id=PROJECT_ID,
    ),
    ReferenceSpace(
        id=STANDARD_REFERENCE_SPACE_ID,
        name="Minnie65 standard-transform coordinates",
        description="Minnie standard-transform coordinates in microns.",
        unit=Unit.MICRONS_LENGTH,
        default_2d_view=default_view,
        anatomical_orientation=AnatomicalOrientation(
            medial_to_lateral=SignedAxis.PLUS_X,
            dorsal_to_ventral=SignedAxis.PLUS_Y,
            rostral_to_caudal=SignedAxis.MINUS_Z,
            pia_to_white_matter=SignedAxis.PLUS_Y,
        ),
        project_id=PROJECT_ID,
    ),
]

spatial_locations = []
for row in positions.to_dict("records"):
    common = {
        "dataitem_id": row["dataitem_id"],
        "location_type": LocationType.SOMA,
        "project_id": PROJECT_ID,
    }
    spatial_locations.extend(
        [
            SpatialLocation(
                reference_space=CAVE_REFERENCE_SPACE_ID,
                x=float(row["pt_position_x"]),
                y=float(row["pt_position_y"]),
                z=float(row["pt_position_z"]),
                **common,
            ),
            SpatialLocation(
                reference_space=STANDARD_REFERENCE_SPACE_ID,
                x=float(row["x_medial-lateral"]),
                y=float(row["y_dorsal-ventral"]),
                z=float(row["z_caudal-rostral"]),
                **common,
            ),
        ]
    )

In [5]:
key_columns = ["project_id", "dataitem_id", "reference_space", "location_type"]
expected = pd.DataFrame([row.model_dump(mode="json") for row in spatial_locations])
expected = expected.set_index(key_columns).sort_index()
before = read_spatial_locations(
    PROJECT_ID,
    reference_spaces=[CAVE_REFERENCE_SPACE_ID, STANDARD_REFERENCE_SPACE_ID],
    output_root=OUTPUT_ROOT,
).to_pandas().set_index(key_columns).sort_index()
assert before.index.is_unique, "Existing spatial keys contain duplicates; stop and investigate."
assert expected.index.equals(before.index), (
    "Incoming keys differ from the existing release. Check the output root and neuron roster; "
    "no writes have been made."
)
print(f"Validated {len(expected):,} existing spatial rows; no new keys will be inserted.")

Validated 162,118 existing spatial rows; no new keys will be inserted.


## Compare with the existing v6 release (read only)

Read `scratch/ccm_release_202610_v6` explicitly, regardless of the configured write target. Compare the two Minnie reference spaces with the incoming rows. A normal first repair has unchanged row counts and keys, unchanged CAVE coordinates, and standard-transform coordinates 1,000 times larger. After repair, rerunning this comparison should show no changed coordinates.

In [6]:
from IPython.display import display

V6_ROOT = REPO_ROOT / "scratch/ccm_release_202610_v6"
v6_before = read_spatial_locations(
    PROJECT_ID,
    reference_spaces=[CAVE_REFERENCE_SPACE_ID, STANDARD_REFERENCE_SPACE_ID],
    output_root=V6_ROOT,
).to_pandas().set_index(key_columns).sort_index()
print(f"Comparison source: {V6_ROOT}")
print(f"Configured write target: {OUTPUT_ROOT}")

comparison_rows = []
for space in [CAVE_REFERENCE_SPACE_ID, STANDARD_REFERENCE_SPACE_ID]:
    existing = v6_before.loc[v6_before.index.get_level_values("reference_space") == space]
    incoming = expected.loc[expected.index.get_level_values("reference_space") == space]
    comparison_rows.append({
        "reference_space": space,
        "v6_rows": len(existing),
        "incoming_rows": len(incoming),
        "v6_unique_cells": existing.index.get_level_values("dataitem_id").nunique(),
        "incoming_unique_cells": incoming.index.get_level_values("dataitem_id").nunique(),
        "v6_duplicate_keys": int(existing.index.duplicated().sum()),
        "incoming_duplicate_keys": int(incoming.index.duplicated().sum()),
        "new_keys": len(incoming.index.difference(existing.index)),
        "v6_keys_not_in_incoming": len(existing.index.difference(incoming.index)),
    })
display(pd.DataFrame(comparison_rows).set_index("reference_space"))
assert v6_before.index.is_unique and expected.index.is_unique, "Duplicate comparison keys."

axes = ["x", "y", "z"]
matched_keys = v6_before.index.intersection(expected.index)
changed = ~np.isclose(
    v6_before.loc[matched_keys, axes], expected.loc[matched_keys, axes], rtol=1e-12, atol=1e-12,
).all(axis=1)
changed_rows = pd.Series(changed, index=matched_keys, name="changed_coordinate_rows")
display(changed_rows.groupby(level="reference_space").agg(["size", "sum"]).rename(
    columns={"size": "matched_rows", "sum": "changed_coordinate_rows"},
))
display(pd.concat({"existing_v6": v6_before, "incoming": expected}, names=["source"])
    .groupby(["source", "reference_space"])[axes].agg(["min", "max"]))

Comparison source: /workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v6
Configured write target: /workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v6


,v6_rows,incoming_rows,v6_unique_cells,incoming_unique_cells,v6_duplicate_keys,incoming_duplicate_keys,new_keys,v6_keys_not_in_incoming
reference_space,,,,,,,,
minnie65_v1822_cave_nm,81059,81059,81059,81059,0,0,0,0
minnie65_v1822_standard_transform_um,81059,81059,81059,81059,0,0,0,0


,matched_rows,changed_coordinate_rows
reference_space,,
minnie65_v1822_cave_nm,81059,0
minnie65_v1822_standard_transform_um,81059,81059


x                \
                                                            min           max   
source      reference_space                                                     
existing_v6 minnie65_v1822_cave_nm                219776.000000  1.677952e+06   
            minnie65_v1822_standard_transform_um       0.166618  1.617981e+00   
incoming    minnie65_v1822_cave_nm                219776.000000  1.677952e+06   
            minnie65_v1822_standard_transform_um     166.617657  1.617981e+03   

                                                              y                \
                                                            min           max   
source      reference_space                                                     
existing_v6 minnie65_v1822_cave_nm                293504.000000  1.234496e+06   
            minnie65_v1822_standard_transform_um      -0.003373  8.919456e-01   
incoming    minnie65_v1822_cave_nm                293504.000000  1.234496e+06   
            minnie65_v1822_standard_transform_um      -3.372603  8.919456e+02   

                                                            z                
                                                          min           max  
source      reference_space                                                  
existing_v6 minnie65_v1822_cave_nm                594400.0000  1.113840e+06  
            minnie65_v1822_standard_transform_um       0.5944  1.113840e+00  
incoming    minnie65_v1822_cave_nm                594400.0000  1.113840e+06  
            minnie65_v1822_standard_transform_um     594.4000  1.113840e+03

## Merge CCM rows and verify

The next cell changes the configured release. Verify the comparison and printed target above before running it. After successful verification, run the separate cache-save cell below. Writes to the two Delta tables and cache are separate operations; if interrupted, rerun this notebook.

In [7]:
print(write_models(reference_spaces, output_root=OUTPUT_ROOT))
print(write_models(spatial_locations, output_root=OUTPUT_ROOT))

after = read_spatial_locations(
    PROJECT_ID,
    reference_spaces=[CAVE_REFERENCE_SPACE_ID, STANDARD_REFERENCE_SPACE_ID],
    output_root=OUTPUT_ROOT,
).to_pandas().set_index(key_columns).sort_index()
assert after.index.is_unique and after.index.equals(before.index), "Spatial row keys changed."
np.testing.assert_allclose(after[["x", "y", "z"]], expected[["x", "y", "z"]], rtol=0, atol=0)
print(f"Verified {len(after):,} rows: corrected coordinates, unchanged keys, no duplicates.")

WrittenResult(class_name='ReferenceSpace', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v6/referencespace'), mode='merge_scoped', predicates=('(target."project_id" IS NOT DISTINCT FROM source."project_id") AND target."id" = source."id"',), rows_written=0)
WrittenResult(class_name='SpatialLocation', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v6/spatiallocation'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."dataitem_id" = source."dataitem_id" AND target."reference_space" = source."reference_space" AND target."location_type" = source."location_type" AND target."project_id" IN (\'minnie65\')',), rows_written=81059)
Verified 162,118 rows: corrected coordinates, unchanged keys, no duplicates.


## Save the corrected CAVE nucleus-position cache

After the CCM write and verification above, save the full deduplicated nucleus table (not just the neuron subset) to the same file as `EM_minnie1822_cave_query.ipynb`. As in that notebook, omit `volume` from the positions cache.

In [8]:
DATA_ROOT.mkdir(parents=True, exist_ok=True)
cache_path = DATA_ROOT / "minnie_v1822_nucleus_positions.parquet"
nuc_df.drop(columns="volume").to_parquet(cache_path)
print(f"Saved {len(nuc_df):,} nucleus positions in nm and transformed microns to {cache_path}")

Saved 118,464 nucleus positions in nm and transformed microns to /workspaces/ConnectsCommonConnectivity/data/EM minnie 1822/minnie_v1822_nucleus_positions.parquet
